# Detecting Illicit Ethereum Transactions — Exploratory Data Analysis**DCS 404 — Machine Learning and Artificial Intelligence**Sanskriti Acharya and Sweta SharmaThis notebook explores the dataset we built and records the preprocessing decisionswe made, with the reasoning behind each one.A note on where the data comes from, because it shapes everything below. We took**only labels** from an external source: the MyEtherWallet community darklist ofreported phishing and scam addresses. Every **feature** in this notebook wasderived by us from raw transaction data fetched through the Blockscout GraphQLAPI, and the transaction graph was built by us in NetworkX.The modelling itself lives in `02_modeling.ipynb`. This notebook is aboutunderstanding the data first.

In [ ]:
import sysfrom pathlib import PathROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT))import matplotlib.pyplot as pltimport numpy as npimport pandas as pdimport seaborn as snsfrom src.features import BEHAVIOUR_FEATURESfrom src.graph import build_graph, graph_summarysns.set_theme(style="whitegrid", palette="muted")plt.rcParams["figure.dpi"] = 110pd.set_option("display.width", 120)pd.set_option("display.max_columns", 50)FIGURES = ROOT / "reports" / "figures"FIGURES.mkdir(parents=True, exist_ok=True)

## 1. The dataset`features_raw.csv` holds one row per address: the behavioural features we computedfrom that address's raw transactions.

In [ ]:
features = pd.read_csv(ROOT / "data" / "processed" / "features_raw.csv")print(f"{len(features)} addresses, {len(BEHAVIOUR_FEATURES)} behavioural features")print()print(features["label"].value_counts().rename({0: "licit", 1: "illicit"}))print()print(f"Positive (illicit) rate: {features['label'].mean():.1%}")

### Class balance, and why accuracy is the wrong metricThe dataset is imbalanced but not extremely so. That single number decides how weevaluate everything later: a model that predicts "licit" for every address isalready right about three quarters of the time while being completely useless.So accuracy is reported for completeness only. We select models on **F1 for theillicit class**, which balances catching scams (recall) against not flooding ananalyst with false alarms (precision).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))counts = features["label"].map({0: "licit", 1: "illicit"}).value_counts()axes[0].bar(counts.index, counts.values, color=["#4c72b0", "#c44e52"])axes[0].set_title("Addresses per class")axes[0].set_ylabel("addresses")for i, v in enumerate(counts.values):    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")majority = 1 - features["label"].mean()axes[1].bar(["always predict licit"], [majority], color="#8c8c8c")axes[1].axhline(1.0, ls="--", c="k", lw=1)axes[1].set_ylim(0, 1.05)axes[1].set_title("Accuracy of a useless model")axes[1].text(0, majority, f"{majority:.1%}", ha="center", va="bottom")plt.tight_layout()plt.savefig(FIGURES / "class_balance.png", bbox_inches="tight")plt.show()

## 2. Data qualityTwo things can go wrong in collection, and both need to be visible rather thanquietly dropped.**Failed fetches.** Some addresses time out. The API struggles with veryhigh-volume accounts (one exchange wallet we tried has 784,000 transactions), sothe collector caps how far it will page and records a failure instead of hanging.**Genuinely empty addresses.** Some addresses in the darklist were reported butnever actually transacted, or their activity is not visible through this endpoint.They are real data, not errors, but they carry no signal.

In [ ]:
status_counts = features["status"].value_counts()empty = (features["n_tx"] == 0).sum()print("Collection status:")print(status_counts.to_string())print()print(f"Addresses with zero transactions: {empty} ({empty / len(features):.1%})")print()print("Zero-transaction addresses by class:")print(    features[features["n_tx"] == 0]["label"]    .map({0: "licit", 1: "illicit"})    .value_counts()    .to_string())

**Decision.** Addresses with no transactions are kept, with their features atzero, rather than deleted.Deleting them would be convenient but dishonest: in the real task, an analysthanded an address with no visible history still needs an answer, and "we have noevidence" is a legitimate state for the model to learn. Removing them would alsochange the class balance, making the reported scores describe an easier problemthan the one we claim to solve.

In [ ]:
print("Transactions collected per address:")print(features["n_tx"].describe().round(2).to_string())fig, axes = plt.subplots(1, 2, figsize=(11, 4))axes[0].hist(features["n_tx"], bins=40, color="#4c72b0")axes[0].set_title("Transactions per address")axes[0].set_xlabel("transactions")axes[0].set_ylabel("addresses")for label, name, colour in [(0, "licit", "#4c72b0"), (1, "illicit", "#c44e52")]:    subset = features[features["label"] == label]["n_tx"]    axes[1].hist(subset, bins=30, alpha=0.6, label=name, color=colour, density=True)axes[1].set_title("Transactions per address, by class")axes[1].set_xlabel("transactions")axes[1].legend()plt.tight_layout()plt.savefig(FIGURES / "tx_counts.png", bbox_inches="tight")plt.show()

The collector caps each address at 8 pages of 8 transactions, so 64 is the ceilingby design, not a property of the addresses. That cap exists because of the API'srate limit, and it means the features describe an address's **most recent 64transactions** rather than its entire history. This is a real limitation and it isstated as one in the report.

## 3. The era checkThis is the most important plot in the notebook, because it validates the way webuilt the dataset.669 of the 715 darklist entries are dated 2017 or 2018. If we had sampled ourlicit addresses from recent blocks — the obvious, easy thing to do — the twoclasses would sit in completely different block ranges. A model could thenseparate them perfectly using block number alone, scoring beautifully while havinglearned nothing except that old addresses are scams.We instead sampled licit addresses from blocks **2,912,407 to 6,988,615**, whichis exactly 2017 and 2018. The two distributions below should overlap. If they donot, every result that follows is worthless.

In [ ]:
active = features[features["first_block"] > 0]fig, axes = plt.subplots(1, 2, figsize=(12, 4))for label, name, colour in [(0, "licit", "#4c72b0"), (1, "illicit", "#c44e52")]:    subset = active[active["label"] == label]["first_block"]    axes[0].hist(subset, bins=50, alpha=0.6, label=name, color=colour, density=True)axes[0].set_title("First block seen, by class")axes[0].set_xlabel("block number")axes[0].legend()data = [active[active["label"] == v]["first_block"] for v in (0, 1)]axes[1].boxplot(data, tick_labels=["licit", "illicit"], vert=False)axes[1].set_title("First block seen — overlap check")axes[1].set_xlabel("block number")plt.tight_layout()plt.savefig(FIGURES / "era_overlap.png", bbox_inches="tight")plt.show()print("Median first block:")print(active.groupby("label")["first_block"].median().rename({0: "licit", 1: "illicit"}).to_string())overlap_lo = max(active[active.label == 0].first_block.quantile(0.05),                 active[active.label == 1].first_block.quantile(0.05))overlap_hi = min(active[active.label == 0].first_block.quantile(0.95),                 active[active.label == 1].first_block.quantile(0.95))print(f"\nCentral 90% ranges overlap between blocks {overlap_lo:,.0f} and {overlap_hi:,.0f}")

**Decision.** Block numbers are kept in the dataset because the temporal splitneeds them, but `first_block` and `last_block` are **excluded from the features anymodel is allowed to see**. `src/features.py` enforces this: `BEHAVIOUR_FEATURES`lists what a model may use, and the block columns are not in it.Even with the eras matched, handing a model an absolute timestamp invites it tomemorise periods rather than learn behaviour. Durations derived from blocks —`lifetime_days`, `mean_days_between_tx` — are kept, because a lifetime means thesame thing regardless of when it happened.

## 4. What separates the two classes?Now the actual exploration. Most of these features are heavily skewed — a handfulof addresses move thousands of ETH while most move a fraction of one — so thevalue plots use a log scale.

In [ ]:
interesting = [    "n_tx", "n_counterparties", "counterparty_ratio", "value_concentration",    "lifetime_days", "tx_per_active_day", "burstiness", "mean_gas_price_gwei",    "total_eth_received", "total_eth_sent", "sent_ratio", "zero_value_ratio",]fig, axes = plt.subplots(3, 4, figsize=(16, 10))for ax, name in zip(axes.flat, interesting):    for label, cname, colour in [(0, "licit", "#4c72b0"), (1, "illicit", "#c44e52")]:        values = features[features["label"] == label][name]        if values.max() > 100:            values = np.log1p(values)            ax.set_xlabel("log(1 + value)")        ax.hist(values, bins=30, alpha=0.55, label=cname, color=colour, density=True)    ax.set_title(name, fontsize=10)    ax.tick_params(labelsize=8)axes.flat[0].legend(fontsize=8)plt.tight_layout()plt.savefig(FIGURES / "feature_distributions.png", bbox_inches="tight")plt.show()

In [ ]:
# Rank features by how far apart the two classes sit, in units of pooled spread.# This is a standardised mean difference, so features on wildly different scales# can be compared directly.rows = []for name in BEHAVIOUR_FEATURES:    licit = features[features["label"] == 0][name]    illicit = features[features["label"] == 1][name]    pooled = np.sqrt((licit.var() + illicit.var()) / 2)    if pooled > 0:        rows.append({            "feature": name,            "licit_mean": licit.mean(),            "illicit_mean": illicit.mean(),            "separation": (illicit.mean() - licit.mean()) / pooled,        })separation = pd.DataFrame(rows).reindex(    pd.DataFrame(rows)["separation"].abs().sort_values(ascending=False).index)separation.head(15).round(3)

In [ ]:
top = separation.head(12).iloc[::-1]colours = ["#c44e52" if v > 0 else "#4c72b0" for v in top["separation"]]plt.figure(figsize=(8, 5))plt.barh(top["feature"], top["separation"], color=colours)plt.axvline(0, c="k", lw=1)plt.xlabel("standardised difference  (positive = higher for illicit)")plt.title("Which features separate illicit from licit addresses?")plt.tight_layout()plt.savefig(FIGURES / "feature_separation.png", bbox_inches="tight")plt.show()

## 5. Correlation between featuresSeveral features are built from the same underlying quantities, so someredundancy is expected. It is worth seeing how much, because heavily correlatedinputs make a logistic regression's coefficients hard to interpret — andinterpretability is part of what this project is for.

In [ ]:
correlations = features[BEHAVIOUR_FEATURES].corr()plt.figure(figsize=(13, 10))sns.heatmap(correlations, cmap="RdBu_r", center=0, square=True,            cbar_kws={"shrink": 0.7}, xticklabels=True, yticklabels=True)plt.title("Correlation between behavioural features")plt.xticks(fontsize=7, rotation=90)plt.yticks(fontsize=7)plt.tight_layout()plt.savefig(FIGURES / "correlation_heatmap.png", bbox_inches="tight")plt.show()pairs = correlations.abs().where(np.triu(np.ones(correlations.shape), k=1).astype(bool)).stack()print("Most correlated feature pairs:")print(pairs.sort_values(ascending=False).head(10).round(3).to_string())

**Decision.** Correlated features are kept rather than pruned.Random forests are unbothered by correlation, and for the logistic regression westandardise inputs and use L2 regularisation, which shares weight betweencorrelated inputs instead of letting one blow up. Dropping features by hand wouldcost us interpretable columns for very little gain — and we would rather explain`n_sent` and `sent_ratio` separately to an examiner than explain why one of themvanished.

## 6. The transaction graphEverything so far describes an address in isolation. The project's actual questionis whether an address's **connections** reveal fraud that its own behaviour doesnot.The graph is built from the transactions we collected: each address is a node andeach transfer a directed edge. It is much larger than the labelled set, because italso contains every counterparty our labelled addresses touched — which is exactlythe context the graph features exploit.

In [ ]:
edges = pd.read_csv(ROOT / "data" / "processed" / "edges.csv")graph = build_graph(edges)summary = graph_summary(graph)for key, value in summary.items():    print(f"  {key}: {value:,.4f}" if isinstance(value, float) else f"  {key}: {value:,}")labelled = set(features["address"])print(f"\n  labelled addresses: {len(labelled):,}")print(f"  discovered counterparties: {summary['nodes'] - len(labelled & set(graph.nodes)):,}")

In [ ]:
degrees = [d for _, d in graph.to_undirected().degree()]fig, axes = plt.subplots(1, 2, figsize=(11, 4))axes[0].hist(degrees, bins=60, color="#4c72b0")axes[0].set_yscale("log")axes[0].set_title("Degree distribution")axes[0].set_xlabel("degree")axes[0].set_ylabel("nodes (log)")axes[1].hist(np.log1p(degrees), bins=50, color="#55a868")axes[1].set_title("Degree distribution (log scale)")axes[1].set_xlabel("log(1 + degree)")plt.tight_layout()plt.savefig(FIGURES / "degree_distribution.png", bbox_inches="tight")plt.show()print(f"Mean degree {np.mean(degrees):.2f}, median {np.median(degrees):.0f}, max {max(degrees):,}")

The degree distribution is heavy-tailed, which is what blockchain graphs normallylook like: most addresses have a couple of counterparties and a few hubs havethousands. That shape is why `pagerank` and `degree` are worth computing — theylocate an address on this spectrum.

## 7. Graph features by class`features_full.csv` is written by `src/train.py` and contains the graph featuresalongside the behavioural ones.One of these needs care. `neighbour_risk_ratio` asks "what fraction of my labelledneighbours are illicit?", and if it were computed using all labels it would leakthe answer: two scam addresses that transact with each other would each reveal theother's label. In training it is computed from **training-split labels only**. Thevalues shown here are the ones the model actually saw.

In [ ]:
full_path = ROOT / "data" / "processed" / "features_full.csv"if not full_path.exists():    print("Run `python -m src.train` first to generate features_full.csv")else:    full = pd.read_csv(full_path)    graph_cols = ["degree", "in_degree", "out_degree", "pagerank",                  "clustering", "community_size", "neighbour_risk_ratio",                  "n_labelled_neighbours"]    print(full.groupby("label")[graph_cols].mean().round(4).T.rename(        columns={0: "licit", 1: "illicit"}).to_string())    fig, axes = plt.subplots(2, 4, figsize=(16, 7))    for ax, name in zip(axes.flat, graph_cols):        for label, cname, colour in [(0, "licit", "#4c72b0"), (1, "illicit", "#c44e52")]:            values = full[full["label"] == label][name]            if values.max() > 100:                values = np.log1p(values)                ax.set_xlabel("log(1 + value)")            ax.hist(values, bins=30, alpha=0.55, label=cname, color=colour, density=True)        ax.set_title(name, fontsize=10)        ax.tick_params(labelsize=8)    axes.flat[0].legend(fontsize=8)    plt.tight_layout()    plt.savefig(FIGURES / "graph_features.png", bbox_inches="tight")    plt.show()

## 8. Preprocessing decisions, collectedEverything decided above, in one place, with the reason for each.| Decision | What we did | Why ||---|---|---|| Duplicate labels | Collapsed 715 darklist rows to 652 unique addresses | The same wallet is reported under several campaigns. Left in, one address could land in both train and test, inflating every score. || Address case | Lower-cased everywhere | The darklist mixes checksummed and lower-case spellings of the same address. || Licit class source | Sampled from 2017–2018 blocks, not recent ones | 97% of the darklist is from that era. Sampling recent addresses would let block number separate the classes perfectly. || Contracts | Excluded from both classes | The darklist is all ordinary wallets, so contracts in the licit class would make "has contract code" a giveaway unrelated to fraud. || Empty addresses | Kept, features zeroed | Deleting them would change the class balance and hide a real case the model must handle. || `gasUsed` | Dropped in favour of `gas` | The API returns 0 for `gasUsed` on many ordinary transfers. `gas` is always populated. || Block numbers | Kept for splitting, excluded from features | Absolute time invites memorising eras; durations derived from it are fine. || Correlated features | Kept | Forests are unaffected; the regression is standardised and L2-regularised. Interpretability is worth more than tidiness here. || Scaling | `StandardScaler` inside the saved pipeline | The scaler is part of the model, so the app cannot apply different preprocessing than training did. || Skew | Left as-is, plotted on log scales | Trees do not care about monotone transforms, and raw units stay explainable to a human. || Split | Temporal, by first appearance | A random split can put two wallets from one scam cluster on both sides. || `neighbour_risk_ratio` | Computed from training labels only | Otherwise the feature leaks the test labels directly. |## What we learnedThe behavioural features do separate the classes, but not trivially — which is theright outcome for a project asking whether graph structure adds anything. Ifbehaviour alone were enough, there would be nothing left for the graph features tocontribute, and the research question would answer itself.`02_modeling.ipynb` measures whether they do.